# Décrypter le Transformer : du caractère au concept
### **Mini-projet MSO 3.7 - Apprentissage bayésien et exploration de textes**

**Auteurs :** LOKE Bamishola - WONE Mamoudou

**Corpus choisi :** Molière

---

Ce notebook est directement basé sur le code fourni en cours par monsieur Velcin, Lui-même inspiré de nanoGPT. Il l'étend pour :
- évaluer les modèles avec des métriques quantitatives (PPL, TTR, hallucination)
- améliorer la génération (température, top-k, nucleus, beam search)
- expérimenter la tokenisation BPE

**Plan :**
1. Setup Colab
2. Données & vocabulaire
3. Bigramme (baseline) — code du prof
4. myGPT (Transformer complet) — code du prof
5. Checkpointing
6. Métriques : PPL, TTR, Hallucination
7. Prompting
8. Stratégie A : Température, Top-k, Nucleus
9. Stratégie B : Beam Search
10. Stratégie C : Tokenisation BPE
11. Étude des hyperparamètres
12. Tableau comparatif final

---
## 0. Setup Google Colab  

Dans un premier temps, nous allons téverser le fichier `moliere.txt` du corpus.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'tiktoken', '-q'])

from google.colab import drive
drive.mount('/content/drive')

CORPUS_FILE = '/content/drive/MyDrive/ECL/traitement_de_texte/corpus/moliere.txt'
print(f'Corpus : {CORPUS_FILE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Corpus : /content/drive/MyDrive/ECL/traitement_de_texte/corpus/moliere.txt


---
## 1. Imports & device  

Dans un second temps, nous allons nous assurer de tourner sur du GPU pour accélérer l'entraînement et bénéficier de la puissance de calcul.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math, os, json
import tiktoken

# Sélection automatique du device
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')

torch.manual_seed(1337)

Device : cuda
GPU : Tesla T4


---
## 2. Préparation des données

In [3]:
# Chargement du corpus (nous avons seulement remplacé le  input.txt par CORPUS_FILE chargé prédemment)
with open(CORPUS_FILE, 'r', encoding='utf-8') as file:
    text = file.read()

print(f'Corpus : {CORPUS_FILE}')
print(f'Taille : {len(text):,} caractères')
print(f'Aperçu :\n{text[:300]}')

Corpus : /content/drive/MyDrive/ECL/traitement_de_texte/corpus/moliere.txt
Taille : 1,664,817 caractères
Aperçu :
﻿OEUVRES COMPLÈTES
DE MOLIÈRE




  PREMIÈRE ÉPOQUE

  1645-1658

  PREMIÈRES OEUVRES; ESSAIS DE JEUNESSE ET IMITATIONS
  DE LA COMMEDIA DELL'ARTE


  I.    --   LE MÉDECIN VOLANT, canevas italien.
  II.   --   LA JALOUSIE DU BARBOUILLÉ, canevas italien.
  III. 1653. L'ÉTOURDI, imitation de l'italie


In [4]:
# Construction du vocabulaire (les caractères uniques dans le texte) — codebook
# Nous avons gardé le code fourni.
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'Vocabulaire : {vocab_size} caractères uniques')

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]          # encoder: str -> list of int
decode = lambda l: ''.join([itos[i] for i in l])  # decoder: list of int -> str

# Encodage du dataset complet
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))  # 90% pour le train
train_data = data[:n]
val_data   = data[n:]
print(f'Train : {len(train_data):,} tokens | Val : {len(val_data):,} tokens')

Vocabulaire : 114 caractères uniques
Train : 1,498,335 tokens | Val : 166,482 tokens


### Hyperparamètres globaux
Nous reprenons les hyperparamètres du prof et nous augmentons pour profiter du GPU T4 de Colab.

In [5]:
# ================================================================
# HYPERPARAMÈTRES - augmentés pour T4
# ================================================================
batch_size    = 64   # avant 4
block_size    = 256  # avant 8 : contexte plus long = meilleure cohérence
max_iters     = 5000
eval_interval = 500  # avant 300
learning_rate = 1e-3
eval_iters    = 200
n_embd        = 256  # avant 32 : embeddings plus riches
n_head        = 8    # nombre de têtes d'attention
n_layer       = 6    # nombre de blocs Transformer
dropout       = 0.1  # régularisation (ajouté nous-même)

print('Les hyperparamètres sont bien définis')

Les hyperparamètres sont bien définis


In [6]:
# get_batch - inchangéc, mais paramétrisé pour flexibilité
def get_batch(split, _block_size=None, _batch_size=None):
    """Génère un batch de données (train ou val).
    x : séquence d'entrée (B, T)
    y : séquence cible = x décalée de 1 (B, T)
    """
    bs = _block_size if _block_size else block_size
    bz = _batch_size if _batch_size else batch_size
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - bs, (bz,))
    x  = torch.stack([d[i:i+bs]   for i in ix])
    y  = torch.stack([d[i+1:i+bs+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()  # pas de backpropagation dans cette fonction, on ne fait qu'évaluer
def estimate_loss(model, _block_size=None, _batch_size=None):
    """Estime la loss moyenne sur train et val (nous avons repris le code de base)."""
    out = {}
    model.eval()  # mode évaluation
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, y = get_batch(split, _block_size, _batch_size)
            logits, loss = model(X, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()  # mode entraînement
    return out

print('get_batch et estimate_loss définis')

get_batch et estimate_loss définis


---
## 3. Modèle Bigramme (Baseline)

Nous avons repris le `bigrammeV2.py` tel quel.
Ce modèle n'a pas d'attention : il sert de **baseline** pour mesurer les gains du Transformer.

In [7]:
# ================================================================
# BIGRAMME - code (bigrammeV2.py), repris tel quel
# Seule modification : generate() accepte temperature/top_k/top_p
#   pour pouvoir comparer les stratégies de génération
# ================================================================

class BigramLanguageModel(nn.Module):
    def __init__(self, bigram_block=64):
        super().__init__()
        self._block_size = bigram_block
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # taille fixée à bigram_block, pas block_size global
        self.position_embedding_table = nn.Embedding(bigram_block, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x      = tok_emb + pos_emb
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss    = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self._block_size:]  # tronquer
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs    = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat((idx, next_idx), dim=1)
        return idx

print('BigramLanguageModel défini')

BigramLanguageModel défini


In [8]:
# Entraînement du Bigramme
# On utilise un block_size réduit pour le Bigramme car il n'a pas de troncature
# dans estimate_loss (contrairement à myGPT)
BIGRAM_BLOCK = 64  # < block_size global (256) pour éviter l'erreur CUDA

print('=== Entraînement du Bigramme (baseline) ===')
bigram_model = BigramLanguageModel(bigram_block=BIGRAM_BLOCK).to(device)
print(f'Paramètres : {sum(p.numel() for p in bigram_model.parameters()):,}')

optimizer_bigram = torch.optim.AdamW(bigram_model.parameters(), lr=learning_rate)

for iter in range(3000):
    if iter % eval_interval == 0 or iter == 2999:
        losses = estimate_loss(bigram_model, BIGRAM_BLOCK)
        print(f'iter {iter}: train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}')
        torch.save(bigram_model.state_dict(), 'checkpoint_bigram.pt')
    xb, yb = get_batch('train', BIGRAM_BLOCK)
    logits, loss = bigram_model(xb, yb)
    optimizer_bigram.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_bigram.step()

idx = torch.zeros((1, 1), dtype=torch.long, device=device)
print('\n--- Texte généré par le Bigramme ---')
print(decode(bigram_model.generate(idx, max_new_tokens=500)[0].tolist()))

=== Entraînement du Bigramme (baseline) ===
Paramètres : 74,866
iter 0: train loss 5.0361, val loss 5.0352
iter 500: train loss 2.4154, val loss 2.4187
iter 1000: train loss 2.4064, val loss 2.4068
iter 1500: train loss 2.4019, val loss 2.4106
iter 2000: train loss 2.3999, val loss 2.4052
iter 2500: train loss 2.4005, val loss 2.4088
iter 2999: train loss 2.4014, val loss 2.4092

--- Texte généré par le Bigramme ---




 pont quluronst pou  aue  mesouert méron'u'aitile, us de s IS
 Moie qucons joue  pagt e SMourisourojonest voretapane mesu MONAGARANARUCEnt dé..
  LE.
 prarin. le.
 fiaièra s nu:
 siesi les vouris  anéleutois fac pre   pou  ps, que  ausis quthis aind'arigle ris air  qutt di, Vi;

detisut bin ntt [7] lo, voux er ce mes usopospu'eson otrrême?
   fa  bie, der


  soiaiaist doue me sel LINOUROue l'ais n'ore voire à dé rblaies?
CAhauperspoin, jeçavi LEt voux  anisan s s d'ou vallie, utr,

  le vépl


In [9]:
torch.save(bigram_model.state_dict(), '/content/drive/MyDrive/ECL/traitement_de_texte/real_checkpoint_bigram.pt')
print('Bigramme sauvegardé')

Bigramme sauvegardé


---
## 4. Mini-GPT - myGPT (Transformer complet)

Le code (`myGPTV2.py`) fourni est repris repris tel quel.
Architecture : Head -> MultiHeadAttention -> FeedForward -> Block -> myGPT.
On rend les hyperparamètres configurables (n_layer, n_head, n_embd, block_size)
pour l'étude des hyperparamètres en section 11.

In [10]:
# ================================================================
# CODE myGPTV2.py - repris tel quel
# Seules modifications :
#   - les hyperparamètres (n_embd, block_size, n_head, n_layer)
#     sont passés en argument pour permettre les comparaisons
#   - generate() accepte temperature, top_k, top_p (stratégies A)
#   - dropout ajouté dans Head et FeedForward (bonne pratique)
# ================================================================

# ajout d'une classe pour gérer la self attention
class Head(nn.Module):
    # gère une seule tête d'attention
    def __init__(self, head_size, _n_embd, _block_size, _dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(_n_embd, head_size, bias=False)
        self.query = nn.Linear(_n_embd, head_size, bias=False)
        self.value = nn.Linear(_n_embd, head_size, bias=False)
        # register_buffer : tensor non entraînable (masque causal)
        self.register_buffer('tril', torch.tril(torch.ones(_block_size, _block_size)))
        self.dropout = nn.Dropout(_dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B,T,C), avec C = head_size
        q = self.query(x)  # (B,T,C), avec C = head_size
        v = self.value(x)  # (B,T,C), avec C = head_size
        # calcule le score d'attention (attention maps)
        wei = q @ k.transpose(-2,-1) * C**-0.5  # (B,T,T), **-0.5 = racine carrée
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))  # (B,T,T)
        # on est obligé de couper la matrice pour correspondre à la taille de la séquence
        # car elle peut être plus petite que block_size lors de la génération
        wei = torch.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v  # (B,T,T) @ (B,T,C) --> (B,T,C)
        return out


# classe pour gérer plusieurs têtes d'attention
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, _n_embd, _block_size, _dropout=0.0):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size, _n_embd, _block_size, _dropout) for _ in range(num_heads)]
        )
        self.proj    = nn.Linear(_n_embd, _n_embd)
        self.dropout = nn.Dropout(_dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


# deux couches feedforward exécutées pour chaque token indépendamment
class FeedForward(nn.Module):
    def __init__(self, _n_embd, _dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(_n_embd, 4 * _n_embd),
            nn.ReLU(),
            nn.Linear(4 * _n_embd, _n_embd),
            nn.Dropout(_dropout),
        )

    def forward(self, x):
        return self.net(x)


# classe pour un bloc Transformer
class Block(nn.Module):
    def __init__(self, _n_embd, _n_head, _block_size, _dropout=0.0):
        super().__init__()
        head_size  = _n_embd // _n_head
        self.sa    = MultiHeadAttention(_n_head, head_size, _n_embd, _block_size, _dropout)
        self.ffwd  = FeedForward(_n_embd, _dropout)
        self.ln1   = nn.LayerNorm(_n_embd)
        self.ln2   = nn.LayerNorm(_n_embd)

    def forward(self, x):
        # version avec connexions résiduelles et layer normalization 
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        # à noter que l'ordre exact des opérations (pre-LayerNorm) diffère légèrement
        # du Transformer original mais fonctionne très bien
        return x


class myGPT(nn.Module):
    def __init__(self, _vocab_size=None, _n_embd=None, _n_head=None,
                 _n_layer=None, _block_size=None, _dropout=0.0):
        super().__init__()
        # utilise les hyperparamètres globaux si non spécifiés
        vs  = _vocab_size  if _vocab_size  else vocab_size
        ne  = _n_embd      if _n_embd      else n_embd
        nh  = _n_head      if _n_head      else n_head
        nl  = _n_layer     if _n_layer     else n_layer
        bs  = _block_size  if _block_size  else block_size
        dr  = _dropout     if _dropout     else dropout
        self._block_size = bs

        # les embeddings (lookup table)
        self.token_embedding_table    = nn.Embedding(vs, ne)
        # embedding positionnel
        self.position_embedding_table = nn.Embedding(bs, ne)
        # blocs Transformer empilés + LayerNorm finale
        self.blocks = nn.Sequential(
            *[Block(ne, nh, bs, dr) for _ in range(nl)],
            nn.LayerNorm(ne)
        )
        # couche linéaire pour projeter les embeddings vers le vocabulaire
        self.lm_head = nn.Linear(ne, vs)

    def forward(self, idx, targets=None):
        B, T = idx.shape  # on récupère les dimensions du batch et de la séquence
        tok_emb = self.token_embedding_table(idx)                               # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x       = tok_emb + pos_emb                                             # (B, T, C)
        # appliquer les blocs Transformer
        x       = self.blocks(x)                                                # (B, T, C)
        logits  = self.lm_head(x)                                               # (B, T, vocab_size)

        if targets is None:  # nécessaire pour la génération de texte
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss    = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        # idx est un tensor de forme (B, T)
        for _ in range(max_new_tokens):
            # couper le contexte pour avoir maximum block_size tokens (code du prof)
            idx_cond = idx[:, -self._block_size:]
            logits, _ = self(idx_cond)       # (B, T, C)
            logits = logits[:, -1, :]        # prendre les logits du dernier token (B, C)

            # --- AJOUT : température ---
            logits = logits / temperature

            # --- AJOUT : top-k sampling ---
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            # --- AJOUT : nucleus (top-p) sampling ---
            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                # retirer les tokens dont la proba cumulée dépasse top_p
                sorted_to_remove = cumulative_probs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[sorted_to_remove] = float('-inf')
                logits = torch.zeros_like(logits).scatter_(1, sorted_idx, sorted_logits)

            probs    = F.softmax(logits, dim=-1)            # (B, C)
            next_idx = torch.multinomial(probs, num_samples=1)   # (B, 1)
            idx      = torch.cat((idx, next_idx), dim=1)    # concaténer
        return idx

print('Head, MultiHeadAttention, FeedForward, Block, myGPT définis')

Head, MultiHeadAttention, FeedForward, Block, myGPT définis


---
## 5. Entraînement du myGPT + Checkpointing

On ajoute par rapport au code du prof :
- `torch.save` pour sauvegarder les poids à chaque évaluation
- `load_state_dict` pour reprendre l'entraînement si interrompu

In [25]:
CHECKPOINT_PATH = 'checkpoint_gpt.pt'

print('=== Entraînement du myGPT ===')
model = myGPT().to(device)
m     = model  # alias identique au code du prof
print(f'Paramètres : {sum(p.numel() for p in m.parameters()):,}')

# --- Reprise depuis checkpoint si disponible ---
if os.path.exists(CHECKPOINT_PATH):
    print(f'Reprise depuis : {CHECKPOINT_PATH}')
    m.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

# Entraînement — identique au prof
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(m)
        print(f'iter {iter}: train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}')
        # --- CHECKPOINTING : sauvegarde des poids ---
        torch.save(m.state_dict(), CHECKPOINT_PATH)
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)  # passe forward
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Génération après entraînement — identique au prof
idx = torch.zeros((1, 1), dtype=torch.long, device=device)
print('\n--- Texte généré par myGPT (génération standard) ---')
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))

=== Entraînement du myGPT ===
Paramètres : 4,858,482
iter 0: train loss 4.9020, val loss 4.9077
iter 500: train loss 1.6363, val loss 1.6885
iter 1000: train loss 1.2724, val loss 1.3931
iter 1500: train loss 1.1349, val loss 1.2992
iter 2000: train loss 1.0550, val loss 1.2513
iter 2500: train loss 1.0077, val loss 1.2328
iter 3000: train loss 0.9636, val loss 1.2186
iter 3500: train loss 0.9312, val loss 1.2144
iter 4000: train loss 0.8965, val loss 1.2092
iter 4500: train loss 0.8680, val loss 1.2094
iter 4999: train loss 0.8422, val loss 1.2048

--- Texte généré par myGPT (génération standard) ---

il sait aisé si peu jusqu'à l'_Zercier_; il introduision de Molière, par Alain en
  l'exemplaire de Molière, podérabine rayonisée du Saiso deux
  louange Ellia avait pmeuvement le sœur est le sens et merve du
  nature, sur cet avis ou dépit contre l'antitre, et l'ordre au roi, le
tard, ne peut jamais presque c'est un lieu chez le croyant d'un peu
moyen-même. Provez-vous rendre ce lieu da

In [26]:
torch.save(m.state_dict(), '/content/drive/MyDrive/ECL/traitement_de_texte/real_checkpoint_gpt.pt')
print('Modèle sauvegardé sur Drive')    

Modèle sauvegardé sur Drive


In [25]:
# Recharger Bigramme
bigram_model = BigramLanguageModel(bigram_block=BIGRAM_BLOCK).to(device)
bigram_model.load_state_dict(torch.load('/content/drive/MyDrive/ECL/traitement_de_texte/real_checkpoint_bigram.pt'))
bigram_model.eval()
print('Bigramme rechargé')

# Recharger myGPT
m = myGPT().to(device)
m.load_state_dict(torch.load('/content/drive/MyDrive/ECL/traitement_de_texte/real_checkpoint_gpt.pt'))
m.eval()
print('myGPT rechargé')

Bigramme rechargé
myGPT rechargé


In [27]:
# ================================================================
# FIX : top_p (Nucleus) manquant dans generate()
# On redéfinit uniquement la méthode generate() pour les deux modèles
# Les poids chargés ne sont PAS affectés
# ================================================================

def generate_fixed(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -self._block_size:]
        logits, _ = self(idx_cond)
        logits = logits[:, -1, :] / temperature

        # Top-k
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        probs = F.softmax(logits, dim=-1)

        # Top-p (Nucleus) — le fix
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_probs[cumulative - sorted_probs > top_p] = 0.0
            sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
            probs = torch.zeros_like(logits).scatter_(1, sorted_idx, sorted_probs)

        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_idx), dim=1)
    return idx

# Patch des deux modèles sans toucher aux poids
import types
m._block_size = block_size
bigram_model._block_size = BIGRAM_BLOCK

m.generate = types.MethodType(generate_fixed, m)
bigram_model.generate = types.MethodType(generate_fixed, bigram_model)

print("generate() corrigé pour m et bigram_model (top_p actif)")

generate() corrigé pour m et bigram_model (top_p actif)


---
## 6. Métriques d'évaluation

La loss d'entraînement ne suffit pas. On ajoute trois métriques :

**Perplexité (PPL)** : mesure standard en NLP.
$$PPL(X) = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(x_i|x_{<i})\right)$$
Comme la `CrossEntropyLoss` de PyTorch correspond déjà au terme logarithmique,
on a simplement : `PPL = exp(val_loss)`.

**TTR (Type-Token Ratio)** : richesse du vocabulaire généré.
$$TTR = \frac{\text{nb mots uniques}}{\text{nb total de mots}}$$

**Taux d'hallucination** *(optionnel)* : proportion de mots absents du corpus d'entraînement.

In [30]:
@torch.no_grad()
def compute_perplexity(model, _block_size=None):
    """Perplexité = exp(val_loss moyenne).
    La CrossEntropyLoss de PyTorch = -log P en base e,
    donc PPL = exp(loss) directement.
    """
    losses = estimate_loss(model, _block_size)
    ppl    = math.exp(losses['val'])
    return ppl, losses['val']


def type_token_ratio(texte_genere):
    """TTR = nb mots uniques / nb total de mots.
    Proche de 1 : texte varié. Proche de 0 : texte répétitif.
    """
    tokens = texte_genere.split()
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


def taux_hallucination(texte_genere, vocab_reference):
    """Proportion de mots générés absents du vocabulaire d'entraînement."""
    mots = texte_genere.lower().split()
    if not mots:
        return 0.0
    inconnus = [m for m in mots if m not in vocab_reference]
    return len(inconnus) / len(mots)


# Vocabulaire de référence = tous les mots du corpus d'entraînement
vocab_reference = set(decode(train_data.tolist()).lower().split())
print(f'Vocabulaire de référence : {len(vocab_reference):,} mots uniques')


def evaluer_modele(model, label, _block_size=None, n_tokens=800,
                   temperature=1.0, top_k=None, top_p=None):
    """Évalue un modèle sur les 3 métriques et affiche un résumé."""
    ppl, val_loss = compute_perplexity(model, _block_size)
    bs  = _block_size if _block_size else block_size
    idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    with torch.no_grad():
        gen  = model.generate(idx, max_new_tokens=n_tokens,
                              temperature=temperature, top_k=top_k, top_p=top_p)
    texte = decode(gen[0].tolist())
    ttr   = type_token_ratio(texte)
    hall  = taux_hallucination(texte, vocab_reference)

    print(f'\n{"="*55}')
    print(f'  {label}')
    print(f'  Perplexité (PPL)   : {ppl:.2f}')
    print(f'  Val Loss           : {val_loss:.4f}')
    print(f'  TTR (diversité)    : {ttr:.4f}')
    print(f'  Hallucination      : {hall*100:.1f}%')
    print(f'  Extrait généré :\n  {texte[:300]}')
    print(f'{"="*55}')

    return {'label': label, 'ppl': round(ppl,2), 'val_loss': round(val_loss,4),
            'ttr': round(ttr,4), 'hallucination': round(hall,4), 'sample': texte[:800]}

print('Métriques définies')

Vocabulaire de référence : 31,467 mots uniques
Métriques définies


In [31]:
# Évaluation du Bigramme et du myGPT de base
res_bigram  = evaluer_modele(bigram_model, 'Bigramme (baseline)', _block_size=BIGRAM_BLOCK)
res_gpt_std = evaluer_modele(m, 'myGPT standard (T=1.0)')


  Bigramme (baseline)
  Perplexité (PPL)   : 11.04
  Val Loss           : 2.4011
  TTR (diversité)    : 0.9015
  Hallucination      : 81.1%
  Extrait généré :
  
 depoiendi   cupa vantse pandie s   PORILUnentrenté [9] ai re lamais e    ête rd'i qusidenerr.


doimoilis.
se ngrt ffat régaie  e?
 à LEhô   uer De pse foien:
secabant ple dr,  bl'e pllauraia  pa chare, be qus ELPHILISGANANTrese LGALEBesoboner
 jue que  agre e t âmen,
 j'acorirve oin l'uréterd'ier

  myGPT standard (T=1.0)
  Perplexité (PPL)   : 3.34
  Val Loss           : 1.2074
  TTR (diversité)    : 0.7692
  Hallucination      : 13.8%
  Extrait généré :
  
Allez; dites-le contribuer le portrait du XVIIe a si grand elle, puis se
retrouver sa pothysique le succès de trendre de salets de la maison,
que M. Loui éclatera sur le théâtre.

ÉLISE.
Non, non, non, il faut d'une âge liberté sage! Elle m'a bruité que je
détruis ne ne sait point vu. Vous l'avez f


---
## 7. Prompting

On influence la génération en forçant un contexte initial (prompt)
avant de laisser le modèle imaginer la suite.  

Pour Gpt

In [32]:
def generer_avec_prompt(model, prompt, max_new_tokens=400, temperature=1.0, top_k=None, top_p=None):
    """Génère du texte à partir d'un prompt.
    On encode le prompt et on laisse le modèle continuer.
    """
    model.eval()
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        out = model.generate(context, max_new_tokens, temperature=temperature,
                             top_k=top_k, top_p=top_p)
    return decode(out[0].tolist())

# Exemples de prompts sur Molière
# Adapter selon le corpus 
prompts = ['ACTE', 'SCENE', 'DON JUAN']

for p in prompts:
    print(f'\n{"="*55}')
    print(f'PROMPT : "{p}"')
    print('='*55)
    print(generer_avec_prompt(m, p, max_new_tokens=300, temperature=0.9, top_k=50))


PROMPT : "ACTE"
ACTEUR.

J'entends deux qu'il est questionné de la forme que ce mot re
donne qualité.

MASCARILLE.

J'en sais.

MADELON.

Quoi! tu as grand commode d'autres coquettes de celle que tu vois paroître?

SGANARELLE.

Je veux pour toi que, tu me la donnasse-t-on pas, et il se touche maudit, et
que ton cœur, p

PROMPT : "SCENE"
SCENE.

  Par un moment dont j'apprendrois atteinte,
  Et qui veut du partout qui la fait voir ma peine.

  MASCARILLE.

  Fut-il qu'un maître à la sévère est encore
  Ou trop avancé ce titre sens fers ou non mari,
  Et qu'un plus d'appas que cela soit de qualité?
  Pour sa murité du mal qui fait son bie

PROMPT : "DON JUAN"
DON JUAN.

Je vous en passe du monde, et moi, c'est le mot de comme
un moment. Je l'avoue que c'est une fois médecine, et qui n'a pris:

  «Ce seroit, sans autre Égyptiens, et frosse est ridicule.»

  «                           «Bercé par «son poëme!» Ce que
  Prétendez-vous d'un peu l'encens d'autre côté.


Pour le Bigramme

In [33]:
# Prompting sur le Bigramme pour comparaison
print('\n=== PROMPTING BIGRAMME (pour comparaison) ===')
for p in prompts:
    print(f'\n{"="*55}')
    print(f'PROMPT : "{p}"')
    print('='*55)
    print(generer_avec_prompt(bigram_model, p, max_new_tokens=300, temperature=0.9, top_k=50))


=== PROMPTING BIGRAMME (pour comparaison) ===

PROMPT : "ACTE"
ACTE   cauaneutre  faren   in imban fourt r  dunst de qus-vaie    t   s montt ras manss ise_BE qus ces d'a qu'aisabi s plon  appone, ss, qus dux.--ntrallne Mai blel   ll
 ponsi_,    quis ver..
 E.

  ches l'e qusonn delit et  point'ueris  JO pone e fitoufos  sais  s-vounqus a OTout r t quièrsin bire
 mo

PROMPT : "SCENE"
SCENEME.
 vou Datr GILE leue   lan potunenste ss d de  s-let uist pa  piolai den  pren st mbourt sir ds
 t.
 houstaples E.
leu   vourus duert u  coruris,
 is ent at danarietrendrt à rs rir lar  ese de.
 far ondena r  euint. entin'elagnd d'ais
 ALE, ucharix, laussen ceuen faurtroitraisel'ade.   prentés-it

PROMPT : "DON JUAN"
DON JUAN va  pa  SIs   ou'yprele  t des Pouspa ce époù  are  mmou'eu, mavoundantoraule le  qurst ne ue DONNEnte LCÈN'ar  voitouie AREoue! quss.

  qus Érruie meser cèrde birs'elusaie, iomore s je




 pouene tendene plt sti! echôtti  les  bi  Donuns  couvoit oin qu'ouire peutôteren  it

---
## 8. Stratégie A — Température, Top-k, Nucleus (Top-p)

La génération par défaut (sampling avec T=1.0) est souvent imparfaite :
- trop déterministe → texte répétitif
- trop aléatoire → texte incohérent

On implémente trois techniques de contrôle (déjà intégrées dans `generate()`) :

- **Température** : divise les logits avant le softmax.
  T < 1 concentre les probabilités (plus déterministe), T > 1 les étale (plus créatif).

- **Top-k** : on ne tire qu'among les k tokens les plus probables,
  les autres sont mis à −∞. Évite les tokens improbables et incohérents.

- **Nucleus (Top-p)** : on tire parmi le plus petit ensemble de tokens
  dont la somme des probabilités dépasse p (ex : 0.9).
  Plus adaptatif que top-k car le nombre de tokens varie selon la distribution.
  C'est la méthode utilisée par GPT-4.

In [34]:
strategies = [
    {'label': 'Standard (T=1.0)',              'temperature': 1.0, 'top_k': None, 'top_p': None},
    {'label': 'Température basse (T=0.5)',     'temperature': 0.5, 'top_k': None, 'top_p': None},
    {'label': 'Température haute (T=1.5)',     'temperature': 1.5, 'top_k': None, 'top_p': None},
    {'label': 'Top-k (k=10)',                  'temperature': 1.0, 'top_k': 10,   'top_p': None},
    {'label': 'Top-k (k=50)',                  'temperature': 1.0, 'top_k': 50,   'top_p': None},
    {'label': 'Nucleus (p=0.9)',               'temperature': 1.0, 'top_k': None, 'top_p': 0.9},
    {'label': 'Nucleus (p=0.95) + T=0.8',     'temperature': 0.8, 'top_k': None, 'top_p': 0.95},
]

sampling_results = []
idx0 = torch.zeros((1, 1), dtype=torch.long, device=device)

print(f"{'Stratégie':<35} {'TTR':>6}  {'Hall.':>6}")
print('-'*52)

for s in strategies:
    with torch.no_grad():
        gen  = m.generate(idx0, max_new_tokens=700,
                          temperature=s['temperature'],
                          top_k=s['top_k'], top_p=s['top_p'])
    texte = decode(gen[0].tolist())
    ttr   = type_token_ratio(texte)
    hall  = taux_hallucination(texte, vocab_reference)
    ppl, vl = compute_perplexity(m)
    res = {'label': s['label'], 'ppl': round(ppl,2), 'val_loss': round(vl,4),
           'ttr': round(ttr,4), 'hallucination': round(hall,4), 'sample': texte[:800]}
    sampling_results.append(res)
    print(f"{s['label']:<35} {ttr:>6.4f}  {hall:>6.4f}")

Stratégie                              TTR   Hall.
----------------------------------------------------
Standard (T=1.0)                    0.7217  0.0522
Température basse (T=0.5)           0.8909  0.0182
Température haute (T=1.5)           0.9688  0.4167
Top-k (k=10)                        0.8558  0.1923
Top-k (k=50)                        0.8144  0.1856
Nucleus (p=0.9)                     0.8158  0.1140
Nucleus (p=0.95) + T=0.8            0.8053  0.1504


In [35]:
# Extraits pour comparaison qualitative
for r in sampling_results:
    print(f'\n--- {r["label"]} ---')
    print(r['sample'][:300])


--- Standard (T=1.0) ---


Ah! Lucas! l'usage, (Il disait.) Mais, tout le baille bon.) Allons, mêle,
à vous dire le plus maudit.

MASCARILLE.

Je vous demeurerai.

MADELON.

Il est vrai, monsieur Gorgibus.

  Le mieux, ou cela qui de sa voir sa manière.

MASCARILLE.

Je ne sais pas d'y aimer jusqu'ici flatte.

VALÈRE.

Quoi

--- Température basse (T=0.5) ---

  Pour ce que vous m'avez dit, ma colère est tout vous.
  Il faut que le temps se vouloit se faire enflat;
  Et ce n'est pas qu'un amour qui pour soi sa flamme,
  Et qui m'oblige à rendre contre cela je me plains.

  ARSINOÉ.

  Monsieur, laissez-moi mon courroux soupçon.

  AGNÈS.

  Non. Mais je 

--- Température haute (T=1.5) ---

  J'ose! prière...

  ELVIRE, comment.

          Tout frabile sur le même retouble.

  ÉRASTEORILLE.

  Il'avoit-fêtu, iant pitié déjà, Conjure;
  Sous six étoiendant ainsi la irnjure.
  Est-ce bon dans là-dessus donnée d'elle.

    Lucinde, payage, à la fens contre Sganque surpre?

  AGNÈS.

  El



---
## 9. Stratégie B — Beam Search

La génération gloutonne (greedy) choisit toujours le token le plus probable à chaque pas.
Elle peut manquer de meilleures séquences globales.

Le **beam search** maintient les `beam_width` meilleures séquences en parallèle
et retourne celle dont le score log-probabilité cumulé est le plus élevé.

- Avantage : séquences plus cohérentes grammaticalement
- Inconvénient : `beam_width` fois plus coûteux, tendance à la répétition

In [36]:
@torch.no_grad()
def beam_search(model, max_new_tokens=300, beam_width=5, prompt=None):
    """Beam search : explore beam_width séquences en parallèle.

    À chaque étape :
    1. Pour chaque faisceau, on calcule les beam_width tokens suivants les plus probables
    2. On obtient beam_width x beam_width candidats
    3. On ne garde que les beam_width meilleurs (score = somme des log-probs)

    Retourne la séquence avec le score final le plus élevé.
    """
    model.eval()
    if prompt:
        start = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    else:
        start = torch.zeros((1, 1), dtype=torch.long, device=device)

    # initialisation : un seul faisceau (score=0, séquence de départ)
    beams = [(0.0, start)]

    for _ in range(max_new_tokens):
        candidats = []
        for score, seq in beams:
            # log-probabilités du token suivant
            logits, _ = model(seq[:, -block_size:])
            log_probs  = F.log_softmax(logits[:, -1, :], dim=-1)  # (1, vocab_size)
            # on garde les beam_width meilleurs tokens
            top_lp, top_ids = torch.topk(log_probs, beam_width)
            for i in range(beam_width):
                nouveau_score = score + top_lp[0, i].item()
                nouvelle_seq  = torch.cat([seq, top_ids[:, i:i+1]], dim=1)
                candidats.append((nouveau_score, nouvelle_seq))
        # garder les beam_width meilleurs faisceaux
        candidats.sort(key=lambda x: x[0], reverse=True)
        beams = candidats[:beam_width]

    # retourner la meilleure séquence
    meilleur_score, meilleure_seq = beams[0]
    return decode(meilleure_seq[0].tolist()), meilleur_score


print('=== Génération par Beam Search (beam_width=5) ===')
beam_texte, beam_score = beam_search(m, max_new_tokens=400, beam_width=5)
print(f'Score log-prob : {beam_score:.2f}\n')
print(beam_texte[:500])

ppl_beam, vl_beam = compute_perplexity(m)
res_beam = {
    'label'        : 'myGPT Beam Search (w=5)',
    'ppl'          : round(ppl_beam, 2),
    'val_loss'     : round(vl_beam, 4),
    'ttr'          : round(type_token_ratio(beam_texte), 4),
    'hallucination': round(taux_hallucination(beam_texte, vocab_reference), 4),
    'sample'       : beam_texte[:800]
}

=== Génération par Beam Search (beam_width=5) ===
Score log-prob : -44.47



  ARNOLPHE.

                                                                                                                                                                                                                                                                                                                                                                                                  


---
## 10. Stratégie C - Tokenisation BPE

Notre modèle traite chaque **caractère** comme un token. Les LLM modernes utilisent
le **Byte Pair Encoding (BPE)** qui encode le texte en sous-mots.

On utilise `tiktoken` (OpenAI), avec l'encodage `gpt2` (50 257 tokens).

On réentraîne le même modèle `myGPT` sur ces nouveaux indices,
puis on compare la qualité du texte généré.

La perplexité BPE **ne se compare pas directement** à la PPL caractères
(vocabulaires de tailles très différentes → échelles de log-prob différentes).
On compare donc surtout la **qualité du texte généré** (TTR, hallucination, lecture).

In [37]:
enc_bpe = tiktoken.get_encoding('gpt2')  # tokenizer GPT-2, vocab = 50 257 tokens

data_bpe = enc_bpe.encode(text)

print('=== Comparaison tokenisation caractères vs BPE ===')
print(f'Caractères : {len(text):>10,} tokens | vocab = {vocab_size}')
print(f'BPE        : {len(data_bpe):>10,} tokens | vocab = {enc_bpe.n_vocab:,}')
print(f'Compression BPE : {len(text)/len(data_bpe):.2f}x moins de tokens')

# Exemple concret sur le même extrait
extrait = text[200:300]
print(f'\nMême extrait tokenisé différemment :')
print(f'  Texte        : {extrait!r}')
print(f'  Tokens car.  ({len(encode(extrait)):2d}) : {encode(extrait)}')
print(f'  Tokens BPE   ({len(enc_bpe.encode(extrait)):2d}) : {enc_bpe.encode(extrait)}')

=== Comparaison tokenisation caractères vs BPE ===
Caractères :  1,664,817 tokens | vocab = 114
BPE        :    715,699 tokens | vocab = 50,257
Compression BPE : 2.33x moins de tokens

Même extrait tokenisé différemment :
  Texte        : "II.   --   LA JALOUSIE DU BARBOUILLÉ, canevas italien.\n  III. 1653. L'ÉTOURDI, imitation de l'italie"
  Tokens car.  (100) : [32, 32, 9, 1, 1, 1, 8, 8, 1, 1, 1, 35, 24, 1, 33, 24, 35, 38, 44, 42, 32, 28, 1, 27, 44, 1, 25, 24, 41, 25, 38, 44, 32, 35, 35, 85, 7, 1, 54, 52, 65, 56, 73, 52, 70, 1, 60, 71, 52, 63, 60, 56, 65, 9, 0, 1, 1, 32, 32, 32, 9, 1, 11, 16, 15, 13, 9, 1, 35, 3, 85, 43, 38, 44, 41, 27, 32, 7, 1, 60, 64, 60, 71, 52, 71, 60, 66, 65, 1, 55, 56, 1, 63, 3, 60, 71, 52, 63, 60, 56]
  Tokens BPE   (45) : [3978, 13, 220, 220, 1377, 220, 220, 9131, 449, 1847, 20958, 10008, 35480, 31597, 33, 2606, 8267, 38351, 11, 460, 1990, 292, 340, 42690, 13, 198, 220, 6711, 13, 1467, 4310, 13, 406, 6, 38351, 51, 11698, 17931, 11, 40260, 390, 300, 6, 1287, 

In [38]:
# Données BPE
data_bpe_t = torch.tensor(data_bpe, dtype=torch.long)
n_bpe      = int(0.9 * len(data_bpe_t))
train_bpe  = data_bpe_t[:n_bpe].to(device)
val_bpe    = data_bpe_t[n_bpe:].to(device)

BPE_BLOCK = 64
BPE_BATCH = 64

def get_batch_bpe(split):
    """Batch loader pour les données BPE — même logique que get_batch du prof."""
    d  = train_bpe if split == 'train' else val_bpe
    ix = torch.randint(len(d) - BPE_BLOCK, (BPE_BATCH,))
    x  = torch.stack([d[i:i+BPE_BLOCK]     for i in ix])
    y  = torch.stack([d[i+1:i+BPE_BLOCK+1] for i in ix])
    return x, y

print('Données BPE prêtes')

Données BPE prêtes


In [39]:
# myGPT réentraîné sur BPE
# Modèle plus léger car le vocab est 50 257 (embedding table très large)
model_bpe = myGPT(_vocab_size=enc_bpe.n_vocab, _n_embd=128, _n_head=4,
                   _n_layer=4, _block_size=BPE_BLOCK, _dropout=0.1).to(device)
print(f'myGPT BPE — Paramètres : {sum(p.numel() for p in model_bpe.parameters()):,}')

opt_bpe   = torch.optim.AdamW(model_bpe.parameters(), lr=1e-3)
BPE_ITERS = 3000

print('\n=== Entraînement myGPT BPE ===')
for iter in range(BPE_ITERS):
    if iter % 500 == 0 or iter == BPE_ITERS - 1:
        model_bpe.eval()
        ls = []
        with torch.no_grad():
            for _ in range(eval_iters):
                xb, yb = get_batch_bpe('val')
                _, loss = model_bpe(xb, yb)
                ls.append(loss.item())
        vl = float(np.mean(ls))
        print(f'iter {iter}: val loss {vl:.4f}, PPL {math.exp(vl):.1f}')
        torch.save(model_bpe.state_dict(), 'checkpoint_bpe.pt')
        model_bpe.train()
    xb, yb = get_batch_bpe('train')
    _, loss = model_bpe(xb, yb)
    opt_bpe.zero_grad(set_to_none=True)
    loss.backward()
    opt_bpe.step()

print('Entraînement BPE terminé')

myGPT BPE — Paramètres : 13,716,049

=== Entraînement myGPT BPE ===
iter 0: val loss 11.0203, PPL 61104.5
iter 500: val loss 3.5567, PPL 35.0
iter 1000: val loss 3.2482, PPL 25.7
iter 1500: val loss 3.1381, PPL 23.1
iter 2000: val loss 3.0945, PPL 22.1
iter 2500: val loss 3.0862, PPL 21.9
iter 2999: val loss 3.0505, PPL 21.1
Entraînement BPE terminé


In [40]:
# Sauvegarde finale sur Drive
torch.save(model_bpe.state_dict(), '/content/drive/MyDrive/ECL/traitement_de_texte/real_checkpoint_bpe.pt')
print('BPE sauvegardé sur Drive')

BPE sauvegardé sur Drive


In [41]:
# Génération et évaluation BPE
model_bpe.eval()
with torch.no_grad():
    idx_bpe = torch.zeros((1, 1), dtype=torch.long, device=device)
    gen_bpe = model_bpe.generate(idx_bpe, max_new_tokens=200,
                                  temperature=0.9, top_k=50)[0].tolist()
texte_bpe = enc_bpe.decode(gen_bpe)

print('=== Texte généré par myGPT BPE ===')
print(texte_bpe)

# Métriques BPE
ls_bpe = []
with torch.no_grad():
    for _ in range(eval_iters):
        xb, yb = get_batch_bpe('val')
        _, loss = model_bpe(xb, yb)
        ls_bpe.append(loss.item())
vl_bpe  = float(np.mean(ls_bpe))
ppl_bpe = math.exp(vl_bpe)
ttr_bpe  = type_token_ratio(texte_bpe)
hall_bpe = taux_hallucination(texte_bpe, vocab_reference)

res_bpe = {
    'label'        : 'myGPT BPE (tiktoken gpt2)',
    'ppl'          : round(ppl_bpe, 2),
    'val_loss'     : round(vl_bpe, 4),
    'ttr'          : round(ttr_bpe, 4),
    'hallucination': round(hall_bpe, 4),
    'sample'       : texte_bpe[:800]
}
print(f'\nPPL={ppl_bpe:.2f}  TTR={ttr_bpe:.4f}  Hall={hall_bpe*100:.1f}%')
print('\n⚠ PPL BPE non comparable à PPL caractères (vocabulaires différents).')
print('  On compare la qualité du texte généré : TTR, hallucination, lecture.')

=== Texte généré par myGPT BPE ===
!
  Si si son benêt de ce discours avec moi.

  MASCARILLE.

                Oui.

  SGANARELLE.

                                        C'est parlé si fort
  dans la maison, et vous vouliez à nos intentions;
  Enleurs, dans mes vains et cœurs tous les défauts
  bases siècle.

  LE NOTAIRE DU PALAIS-ROY-BALLET.

C'est une faveur, qui, a parlé n'est point de même chose; et,
  Il avait en repos fâ

PPL=21.25  TTR=0.8966  Hall=10.3%

⚠ PPL BPE non comparable à PPL caractères (vocabulaires différents).
  On compare la qualité du texte généré : TTR, hallucination, lecture.


---
## 11. Étude des hyperparamètres

On mesure l'impact de `block_size`, `n_layer` et `n_head` sur les performances.

In [42]:
configs = [
    {'label': 'Petit  (2 couches, 2 têtes, ctx=64)',  'n_layer':2, 'n_head':2, 'n_embd':64,  'block_size':64},
    {'label': 'Moyen  (4 couches, 4 têtes, ctx=128)', 'n_layer':4, 'n_head':4, 'n_embd':128, 'block_size':128},
    {'label': 'Grand  (6 couches, 8 têtes, ctx=256)', 'n_layer':6, 'n_head':8, 'n_embd':256, 'block_size':256},
]
hyperparam_results = []

for c in configs:
    print(f"\n--- {c['label']} ---")
    m_cfg = myGPT(_n_embd=c['n_embd'], _n_head=c['n_head'],
                  _n_layer=c['n_layer'], _block_size=c['block_size'], _dropout=0.1).to(device)
    n_p = sum(p.numel() for p in m_cfg.parameters())
    print(f'Paramètres : {n_p:,}')

    opt_cfg = torch.optim.AdamW(m_cfg.parameters(), lr=learning_rate)
    for iter in range(2000):
        if iter % 1000 == 0 or iter == 1999:
            losses = estimate_loss(m_cfg, c['block_size'])
            print(f'  iter {iter}: train {losses["train"]:.4f}, val {losses["val"]:.4f}')
        xb, yb = get_batch('train', c['block_size'])
        _, loss = m_cfg(xb, yb)
        opt_cfg.zero_grad(set_to_none=True)
        loss.backward()
        opt_cfg.step()

    ppl, vl = compute_perplexity(m_cfg, c['block_size'])
    idx0  = torch.zeros((1,1), dtype=torch.long, device=device)
    with torch.no_grad():
        texte = decode(m_cfg.generate(idx0, max_new_tokens=400, temperature=0.9, top_k=50)[0].tolist())
    ttr = type_token_ratio(texte)
    hyperparam_results.append({'label': c['label'], 'params': n_p,
                                'ppl': round(ppl,2), 'val_loss': round(vl,4),
                                'ttr': round(ttr,4), 'sample': texte[:400]})
    print(f'  → PPL={ppl:.2f}  TTR={ttr:.4f}')

print('\n=== Résumé hyperparamètres ===')
print(f"{'Config':<45} {'Params':>10} {'PPL':>8} {'TTR':>8}")
print('-'*73)
for r in hyperparam_results:
    print(f"{r['label']:<45} {r['params']:>10,} {r['ppl']:>8.2f} {r['ttr']:>8.4f}")


--- Petit  (2 couches, 2 têtes, ctx=64) ---
Paramètres : 118,514
  iter 0: train 4.9242, val 4.9177
  iter 1000: train 1.8698, val 1.8819
  iter 1999: train 1.6755, val 1.7199
  → PPL=5.60  TTR=0.9412

--- Moyen  (4 couches, 4 têtes, ctx=128) ---
Paramètres : 837,490
  iter 0: train 4.8973, val 4.8833
  iter 1000: train 1.5225, val 1.5860
  iter 1999: train 1.3087, val 1.4225
  → PPL=4.13  TTR=0.9054

--- Grand  (6 couches, 8 têtes, ctx=256) ---
Paramètres : 4,858,482
  iter 0: train 4.9567, val 4.9596
  iter 1000: train 1.2681, val 1.3917
  iter 1999: train 1.0550, val 1.2569
  → PPL=3.51  TTR=0.8730

=== Résumé hyperparamètres ===
Config                                            Params      PPL      TTR
-------------------------------------------------------------------------
Petit  (2 couches, 2 têtes, ctx=64)              118,514     5.60   0.9412
Moyen  (4 couches, 4 têtes, ctx=128)             837,490     4.13   0.9054
Grand  (6 couches, 8 têtes, ctx=256)           4,858,482   

---
## 12. Tableau comparatif final

Synthèse de tous les résultats.

In [43]:
# Évaluations finales des meilleures stratégies (s'il teplait WONE, n'oublie pas, j'ai preparer ce code pour toi, tu n'as plus qu'à le lancer et à remplir les résultats dans le tableau comparatif final)
res_topk    = evaluer_modele(m, 'myGPT Top-k (k=50, T=1.0)',        top_k=50,  temperature=1.0)
res_nucleus = evaluer_modele(m, 'myGPT Nucleus (p=0.95, T=0.8)',    top_p=0.95, temperature=0.8)

tous_resultats = [res_bigram, res_gpt_std, res_topk, res_nucleus, res_beam, res_bpe]

print('\n' + '='*78)
print('TABLEAU COMPARATIF FINAL')
print('='*78)
print(f"{'Modèle / Stratégie':<42} {'PPL':>7} {'Val Loss':>9} {'TTR':>7} {'Hall.%':>7}")
print('-'*78)
for r in tous_resultats:
    print(f"{r['label']:<42} {r['ppl']:>7.2f} {r['val_loss']:>9.4f} "
          f"{r['ttr']:>7.4f} {r['hallucination']*100:>6.1f}%")
print('='*78)
print('PPL BPE non comparable aux autres (vocabulaire différent)')


  myGPT Top-k (k=50, T=1.0)
  Perplexité (PPL)   : 3.34
  Val Loss           : 1.2061
  TTR (diversité)    : 0.8058
  Hallucination      : 10.8%
  Extrait généré :
  
CHARLOTTE.

Puisque donc vous m'avez aussitôt de ma soupçon et une nuit assez lois
immourée des tendres. Tous les hommes, c'est assuré mon invention pour tous vos
offenses ont dit; mais il doit cent ordre-tois aussi, c'est un
objet effet; qu'on garde de tous vos moments fort offérent
qu'ils se mêle

  myGPT Nucleus (p=0.95, T=0.8)
  Perplexité (PPL)   : 3.34
  Val Loss           : 1.2074
  TTR (diversité)    : 0.7737
  Hallucination      : 3.6%
  Extrait généré :
  

  Pour moi, par des armes, mon cœur, la maîtresse.
  Mon père, qu'on peut tout le monde respirer.

  LYCARSIS.

  Moi de son des précipiteuses, et l'on s'est dit
  Ce qu'il fait juger. Mais enfin que le contre fluchement
  Qui puisse me trouver sensible, et que vous doit être si
  Pour compliment 

TABLEAU COMPARATIF FINAL
Modèle / Stratégie                 

In [44]:
# Extraits de textes générés (WONE,  s'il te plait, il faut mettre cette sortie egalement dans le rapport. Je vais le reviser avec toi si je fini les tests avec le code.)
print('\n=== EXTRAITS POUR LE RAPPORT ===')
for r in tous_resultats:
    print(f'\n--- {r["label"]} ---')
    print(r['sample'][:350])


=== EXTRAITS POUR LE RAPPORT ===

--- Bigramme (baseline) ---

 depoiendi   cupa vantse pandie s   PORILUnentrenté [9] ai re lamais e    ête rd'i qusidenerr.


doimoilis.
se ngrt ffat régaie  e?
 à LEhô   uer De pse foien:
secabant ple dr,  bl'e pllauraia  pa chare, be qus ELPHILISGANANTrese LGALEBesoboner
 jue que  agre e t âmen,
 j'acorirve oin l'uréterd'ieres pans,
 a  LIEt sis be  dens,

SA dute tsese jai

--- myGPT standard (T=1.0) ---

Allez; dites-le contribuer le portrait du XVIIe a si grand elle, puis se
retrouver sa pothysique le succès de trendre de salets de la maison,
que M. Loui éclatera sur le théâtre.

ÉLISE.
Non, non, non, il faut d'une âge liberté sage! Elle m'a bruité que je
détruis ne ne sait point vu. Vous l'avez fait toute votre confidence; de
laisser toutes les 

--- myGPT Top-k (k=50, T=1.0) ---

CHARLOTTE.

Puisque donc vous m'avez aussitôt de ma soupçon et une nuit assez lois
immourée des tendres. Tous les hommes, c'est assuré mon invention pour tous vos
offe

In [46]:
# Sauvegarde finale sur Google Drive (résultats + poids)

SAVE_DIR = '/content/drive/MyDrive/ECL/traitement_de_texte/'

import json
export = {
    'corpus'    : CORPUS_FILE,
    'resultats' : tous_resultats,
    'sampling'  : sampling_results,
    'hyperparam': hyperparam_results,
}
with open(SAVE_DIR + 'resultats.json', 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
torch.save(m.state_dict(), SAVE_DIR + 'checkpoint_gpt.pt')
print('Sauvegardé sur Google Drive')

Sauvegardé sur Google Drive
